In [ ]:
OPENWEATHER_API_KEY = ""

In [4]:
from pydantic import BaseModel, Field
from typing import Literal
from langchain.tools import tool
from langchain_ollama import ChatOllama

class WeatherInput(BaseModel):
    """Input for weather queries."""
    location: str = Field(description="City name or coordinates")
    units: Literal["celsius", "fahrenheit"] = Field(
        default="celsius",
        description="Temperature unit preference"
    )
    include_forecast: bool = Field(
        default=False,
        description="Include 5-day forecast"
    )

@tool(args_schema=WeatherInput)
def get_weather(location: str, units: str = "celsius", include_forecast: bool = False) -> str:
    """Get current weather and optional forecast."""
    url = "https://api.openweathermap.org/data/2.5/weather"
    weather_res = requests.get(
        weather_url,
        params={
            "lat": lat,
            "lon": lon,
            "appid": OPENWEATHER_API_KEY,
            "units": "metric",   # 섭씨
            "lang": "kr"
        }
    ).json()
    temp = weather_res["main"]["temp"]
    feels_like = weather_res["main"]["feels_like"]
    desc = weather_res["weather"][0]["description"]
    wind = weather_res["wind"]["speed"]

    result = (
        f"📍 {location} ({country}) 현재 날씨\n"
        f"- 상태: {desc}\n"
        f"- 기온: {temp}°C (체감 {feels_like}°C)\n"
        f"- 풍속: {wind} m/s"
    )

    if include_forecast:
        result += "\n- 예보: OpenWeather 5-day API 연동 가능"

    return result

c:\Users\Bistelligence\AppData\Local\pypoetry\Cache\virtualenvs\ollama-app-PvdVXOUT-py3.11\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [5]:
llm = ChatOllama(
    model="qwen3-vl:8b",
    temperature=0
)

llm_with_tools = llm.bind_tools([get_weather])

In [7]:
response = llm_with_tools.invoke("서울 날씨 알려주고 예보도 포함해")

tool_call = response.tool_calls[0]
result = get_weather.invoke(tool_call["args"])
print(result)

NameError: name 'requests' is not defined

In [ ]:
response = llm_with_tools.invoke("서울 날씨 알려주고 예보도 포함해")

tool_call = response.tool_calls[0]
result = get_weather.invoke(tool_call["args"])
print(result)
